# 00 — The national network and its data

**Goal (PROBLEM.md):** lay out *who is in the federation* and build the *operational panel*
every later decision reads.

This is the national scale-up of `src/`: instead of a top-5-per-region sample of 35 trusts,
the federation is **every NHS trust that reported demand in all 60 months** — no selection
rule, no ranking, no cherry-picking. Independent/private providers are excluded (NHS only).

- Branch hospital ↔ NHS trust (3-char `R`-prefixed org code)
- Board ↔ ICB (the trust's parent at a fixed anchor month)
- Department ↔ treatment function (specialty)
- Window **2021-04 … 2026-03**, frozen: COVID-free (first modelled month is 2022-04) and one
  consistent reporting regime (specialties went 19 → 23 at 2021-04).

In [1]:
import glob
import json
import os
import time
import pandas as pd
import requests

RAW = "../data/nhs_rtt_referral_to_treatment"
WINDOW = ("2021-04", "2026-03")   # frozen window: COVID-free + single reporting regime
ANCHOR = "2025-12"                # fixed month for ICB membership (ICB boundaries moved over the years)
TRUST = r"^R[A-Z0-9]{2}$"         # NHS trusts only -> excludes independent/private providers

files = [p for p in sorted(glob.glob(f"{RAW}/rtt_*.csv"))
         if WINDOW[0] <= os.path.basename(p)[4:11] <= WINDOW[1]]
USE = ["Provider Org Code", "Provider Org Name", "Provider Parent Org Code",
       "RTT Part Type", "Treatment Function Code", "Total All"]

frames, anchor_parent, names = [], None, {}
for path in files:
    month = os.path.basename(path)[4:11]
    df = pd.read_csv(path, usecols=USE, dtype=str)
    df = df[df["Provider Org Code"].str.match(TRUST, na=False)]
    if month == ANCHOR:
        anchor_parent = df.drop_duplicates("Provider Org Code") \
                          .set_index("Provider Org Code")["Provider Parent Org Code"]
    names.update(df.drop_duplicates("Provider Org Code")
                   .set_index("Provider Org Code")["Provider Org Name"].to_dict())
    df = df[(df["RTT Part Type"] == "Part_3") & (df["Treatment Function Code"] != "C_999")]
    df["v"] = pd.to_numeric(df["Total All"], errors="coerce").fillna(0)
    frames.append(df.groupby("Provider Org Code")["v"].sum().rename(month))
    print(month, end=" ", flush=True)

panel = pd.concat(frames, axis=1)
complete = panel.dropna()
complete = complete[(complete > 0).all(axis=1)]   # demand reported in EVERY window month
print(f"\nNHS trusts seen: {len(panel)} -> complete in all {len(files)} months: {len(complete)}")

2021-04 

2021-05 

2021-06 

2021-07 

2021-08 

2021-09 

... (51 outputs left out)

2026-01 

2026-02 

2026-03 


NHS trusts seen: 172 -> complete in all 60 months: 139


## The board layer, resolved live from the NHS ODS registry

Each trust's ICB (its parent at the anchor month) is mapped to its NHS England region by
querying the official Organisation Data Service — no hand-typed mapping anywhere.

In [2]:
icb_of = anchor_parent.reindex(complete.index).dropna()
icbs = sorted(set(icb_of))
ORD = "https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations"

def org(code):
    return requests.get(f"{ORD}/{code}", timeout=20).json()["Organisation"]

icb_region, region_name = {}, {}
for q in icbs:                                    # ICB -> region code (RE2 = "is commissioned by")
    rels = [r["Target"]["OrgId"]["extension"] for r in org(q)["Rels"]["Rel"]
            if r["Status"] == "Active" and r["id"] == "RE2"]
    icb_region[q] = rels[0] if rels else None
    time.sleep(0.15)
for y in sorted({v for v in icb_region.values() if v}):
    region_name[y] = org(y)["Name"]
    time.sleep(0.15)

region_of = {t: region_name.get(icb_region.get(q)) for t, q in icb_of.items()}
missing = [t for t, r in region_of.items() if not r]
print(f"ICBs {len(icbs)} | regions {len(region_name)} | trusts without a region: {len(missing)} {missing}")

ICBs 42 | regions 7 | trusts without a region: 0 []


## The operational panel

One row per `(trust, month, RTT part, department)` with the counts every later notebook reads.
Counts live in **`Total All`** — the raw `Total` column is empty for Part_2/2A/3, so using it
would zero out all demand. The 105 waiting-band columns are compressed to three age buckets.

In [3]:
OUT = "../data/processed/rtt_long_national.csv"
ID = ["Provider Org Code", "Provider Org Name", "RTT Part Type",
      "Treatment Function Code", "Treatment Function Name"]
BANDS = [f"Gt {i:02d} To {i + 1:02d} Weeks SUM 1" for i in range(104)] + ["Gt 104 Weeks SUM 1"]
TOT = ["Total", "Total All"]
codes = set(complete.index)

if os.path.exists(OUT):
    long = pd.read_csv(OUT)
    print(f"panel already built - loaded {len(long):,} rows")
else:
    monthly = []
    for path in files:
        month = os.path.basename(path)[4:11]
        df = pd.read_csv(path, usecols=ID + BANDS + TOT, dtype={c: str for c in ID})
        df = df[df["Treatment Function Code"] != "C_999"]      # drop the double-counting totals row
        df = df[df["Provider Org Code"].isin(codes)]
        df[BANDS + TOT] = df[BANDS + TOT].apply(pd.to_numeric, errors="coerce").fillna(0)
        df = df.assign(weeks_0_18=df[BANDS[:18]].sum(axis=1),
                       weeks_18_52=df[BANDS[18:52]].sum(axis=1),
                       weeks_52_plus=df[BANDS[52:]].sum(axis=1))
        out = df.groupby(ID, as_index=False)[
            ["weeks_0_18", "weeks_18_52", "weeks_52_plus", "Total", "Total All"]].sum()
        out.insert(0, "month", month)
        out.columns = ["month", "provider_code", "provider_name", "part", "specialty_code",
                       "specialty_name", "weeks_0_18", "weeks_18_52", "weeks_52_plus",
                       "total", "total_all"]
        monthly.append(out)
        print(month, end=" ", flush=True)
    long = pd.concat(monthly, ignore_index=True)
    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    long.to_csv(OUT, index=False)
    print(f"\nbuilt {OUT}: {len(long):,} rows")

2021-04 

2021-05 

2021-06 

2021-07 

2021-08 

2021-09 

... (51 outputs left out)

2026-01 

2026-02 

2026-03 


built ../data/processed/rtt_long_national.csv: 628,911 rows


## The org chart, and the panel's integrity

A **unit** = `(trust, department)` with demand in every one of the 60 months — the same
completeness principle applied to trusts, applied one level down.

In [4]:
demand = long[long["part"] == "Part_3"]
months = sorted(long["month"].unique())
seen = demand[demand["total_all"] > 0].groupby(["provider_code", "specialty_code"])["month"].nunique()
units = seen[seen == len(months)].index                      # complete (trust, department) series
unit_df = pd.DataFrame(list(units), columns=["provider_code", "specialty_code"])
per_trust = unit_df.groupby("provider_code")["specialty_code"].apply(list)

network = {"window": {"start": WINDOW[0], "end": WINDOW[1], "months": len(months)},
           "anchor_month": ANCHOR, "selection": "all NHS trusts with demand in every window month",
           "regions": []}
for region in sorted(region_name.values()):
    trusts = [t for t in per_trust.index if region_of.get(t) == region]
    network["regions"].append({"region": region, "trusts": [
        {"code": t, "name": names[t], "icb": icb_of[t], "specialties": len(per_trust[t]),
         "mean_monthly_referrals": int(round(complete.loc[t].mean()))}
        for t in sorted(trusts, key=lambda x: -complete.loc[x].mean())]})
with open("../config/nhs_network_national.json", "w") as f:
    json.dump(network, f, indent=1)

assert set(long["part"]) == {"Part_1A", "Part_1B", "Part_2", "Part_2A", "Part_3"}
assert long.groupby("provider_code")["month"].nunique().eq(len(months)).all(), "a trust is missing months"
n_trusts = len(per_trust)
print(f"panel OK: {len(months)} months ({months[0]}..{months[-1]}) | {long['provider_code'].nunique()} trusts | "
      f"{long['specialty_code'].nunique()} departments | {len(long):,} rows")
print(f"federation: {n_trusts} clients | {len(unit_df):,} units | "
      f"{len(network['regions'])} regions | {icb_of.nunique()} boards")
print(f"units per client: min {per_trust.map(len).min()} | median {int(per_trust.map(len).median())} | "
      f"max {per_trust.map(len).max()}")
print(f"demand spread across clients: {complete.mean(axis=1).max() / complete.mean(axis=1).min():,.0f}x")

panel OK: 60 months (2021-04..2026-03) | 139 trusts | 23 departments | 628,911 rows
federation: 138 clients | 2,076 units | 7 regions | 42 boards
units per client: min 1 | median 18 | max 23
demand spread across clients: 1,464x
